# Daily Delhi Climate — Time-Series Analysis & Forecasting
**Group 2 — Formative 1**

**Dataset:** Daily Delhi Climate Dataset (Kaggle)
**Source:** https://www.kaggle.com/datasets/sumanthvrao/daily-climate-time-series-data
**Target:** Forecast daily mean temperature (`meantemp`) in °C
**Variables:** meantemp, humidity, wind_speed, meanpressure
**Period:** 2013-01-01 → 2017-04-24 (1,575 daily records)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
import pickle, json, os, warnings
warnings.filterwarnings('ignore')

# Load and merge train + test sets
train = pd.read_csv('DailyDelhiClimateTrain.csv', parse_dates=['date'])
test  = pd.read_csv('DailyDelhiClimateTest.csv',  parse_dates=['date'])
df = pd.concat([train, test]).sort_values('date').drop_duplicates(subset=['date']).reset_index(drop=True)

print(f'Shape: {df.shape}')
print(f'Time range: {df.date.min().date()} to {df.date.max().date()}')
print(f'Granularity: Daily')
print(f'Columns: {list(df.columns)}')

In [ ]:
# --- Missing Values ---
print('Missing values per column:')
print(df.isnull().sum())

# Fix pressure outliers (sensor errors: -3.04 and 7679.33 are physically impossible)
q_low  = df.meanpressure.quantile(0.01)
q_high = df.meanpressure.quantile(0.99)
df['meanpressure'] = df['meanpressure'].clip(q_low, q_high)
print(f'\nPressure clipped to [{q_low:.1f}, {q_high:.1f}] hPa (1st–99th percentile)')
print('No missing values found. Outlier clipping applied to meanpressure (sensor errors).')

In [ ]:
# Statistical summary
print('Statistical Summary:')
print(df[['meantemp','humidity','wind_speed','meanpressure']].describe().round(3))

# Distribution plots
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col, color in zip(axes.flatten(),
    ['meantemp','humidity','wind_speed','meanpressure'],
    ['tomato','steelblue','seagreen','mediumpurple']):
    ax.hist(df[col], bins=50, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'Distribution: {col}', fontsize=11)
    ax.axvline(df[col].mean(), color='black', lw=1.5, linestyle='--', label=f'mean={df[col].mean():.1f}')
    ax.legend(fontsize=9)
plt.suptitle('Statistical Distributions of All Variables', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Task 1B — Analytical Questions
### Q1: Does mean temperature show a trend over time?

In [ ]:
monthly = df.set_index('date')['meantemp'].resample('M').mean()
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(monthly.index, monthly.values, lw=1, alpha=0.6, color='steelblue', label='Monthly avg')
ax.plot(monthly.rolling(12).mean(), color='red', lw=2.5, label='12-month MA (annual trend)')
ax.set_title('Q1: Monthly Mean Temperature with 12-Month Moving Average Trend', fontsize=12)
ax.set_ylabel('°C'); ax.legend()
plt.tight_layout(); plt.show()

yearly = df.groupby(df.date.dt.year)['meantemp'].mean().round(2)
print('Yearly mean temperatures:')
print(yearly)

### Q2: Are there lag effects? (Lagged Features)

In [ ]:
df['lag_1d']  = df['meantemp'].shift(1)
df['lag_7d']  = df['meantemp'].shift(7)
df['lag_30d'] = df['meantemp'].shift(30)

df_clean = df.dropna(subset=['lag_1d','lag_7d','lag_30d'])
sample = df_clean.sample(500, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, lag, lbl in zip(axes, ['lag_1d','lag_7d','lag_30d'],
                              ['Lag 1 day','Lag 7 days','Lag 30 days']):
    corr = sample[lag].corr(sample['meantemp'])
    ax.scatter(sample[lag], sample['meantemp'], alpha=0.25, s=12, color='steelblue')
    m, b = np.polyfit(sample[lag], sample['meantemp'], 1)
    x = np.linspace(sample[lag].min(), sample[lag].max(), 100)
    ax.plot(x, m*x+b, 'r-', lw=2)
    ax.set_title(f'Q2: {lbl} | r={corr:.3f}', fontsize=11)
    ax.set_xlabel(f'{lbl} temperature (°C)')
    ax.set_ylabel('meantemp (°C)')
plt.suptitle('Lag Feature Correlations with Mean Temperature', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

for lag in ['lag_1d','lag_7d','lag_30d']:
    r = df_clean[lag].corr(df_clean['meantemp'])
    print(f'{lag:10s} Pearson r = {r:.3f}')

### Q3: How effective are moving averages at capturing the trend?

In [ ]:
df['ma_7d']  = df['meantemp'].rolling(7).mean()
df['ma_30d'] = df['meantemp'].rolling(30).mean()
df['ma_90d'] = df['meantemp'].rolling(90).mean()

subset = df.set_index('date')['2015':]
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(subset.index, subset['meantemp'], alpha=0.25, lw=0.8, color='gray', label='Daily actual')
ax.plot(subset['ma_7d'],  lw=1.5, color='blue',  label='7-day MA')
ax.plot(subset['ma_30d'], lw=2,   color='red',   label='30-day MA')
ax.plot(subset['ma_90d'], lw=2.5, color='green', label='90-day MA')
ax.set_title('Q3: Moving Averages — Trend & Seasonality Decomposition (2015–2017)', fontsize=12)
ax.set_ylabel('°C'); ax.legend()
plt.tight_layout(); plt.show()

### Q4: Do external variables (humidity, wind, pressure) correlate with temperature?

In [ ]:
vars_ = ['humidity','wind_speed','meanpressure']
corrs = {v: round(df[v].corr(df['meantemp']),3) for v in vars_}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, var, color in zip(axes, vars_, ['steelblue','seagreen','mediumpurple']):
    ax.scatter(df[var], df['meantemp'], alpha=0.15, s=8, color=color)
    m, b = np.polyfit(df[var].dropna(), df.loc[df[var].notna(),'meantemp'], 1)
    x = np.linspace(df[var].min(), df[var].max(), 100)
    ax.plot(x, m*x+b, 'r-', lw=2)
    ax.set_title(f'Q4: {var} vs meantemp\nr={corrs[var]}', fontsize=11)
    ax.set_xlabel(var); ax.set_ylabel('meantemp (°C)')
plt.suptitle('External Variable Correlations with Mean Temperature', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print('Pearson correlations with meantemp:')
for k,v in corrs.items(): print(f'  {k:15s}: {v}')

### Q5: What is the seasonal pattern of temperature in Delhi?

In [ ]:
df['month'] = df.date.dt.month
df['year']  = df.date.dt.year

monthly_avg = df.groupby('month')['meantemp'].mean()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Monthly bar chart
axes[0].bar(range(1,13), monthly_avg.values, color='tomato', edgecolor='white', alpha=0.85)
axes[0].set_xticks(range(1,13)); axes[0].set_xticklabels(month_names, fontsize=9)
axes[0].set_title('Q5: Average Temperature by Month', fontsize=11)
axes[0].set_ylabel('Mean Temperature (°C)')

# Yearly heatmap-style line per year
for yr in [2013, 2014, 2015, 2016]:
    yr_data = df[df.year==yr].set_index('date')['meantemp']
    axes[1].plot(yr_data.index.dayofyear, yr_data.values, lw=1.2, alpha=0.7, label=str(yr))
axes[1].set_title('Q5b: Daily Temperature by Year (seasonal overlay)', fontsize=11)
axes[1].set_xlabel('Day of year'); axes[1].set_ylabel('°C'); axes[1].legend()
plt.suptitle('Seasonal Temperature Patterns in Delhi', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

print('Monthly average temperatures (°C):')
for m, t in zip(month_names, monthly_avg.values): print(f'  {m}: {t:.2f}')

## Task 1C — Model Training & Experimental Design

In [ ]:
# Feature engineering
df['day_of_year'] = df.date.dt.dayofyear
df['day_of_week'] = df.date.dt.dayofweek

FEATURES = ['month','day_of_year','day_of_week','humidity','wind_speed','meanpressure',
            'lag_1d','lag_7d','lag_30d','ma_7d','ma_30d']
TARGET = 'meantemp'

df_m = df.dropna(subset=FEATURES).reset_index(drop=True)
X = df_m[FEATURES]; y = df_m[TARGET]

# Time-based 80/20 split (no shuffle — preserves temporal order)
split = int(len(df_m) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_train)
X_te_s = scaler.transform(X_test)

print(f'Training rows : {len(X_train)}')
print(f'Test rows     : {len(X_test)}')
print(f'Train period  : {df_m.iloc[0]["date"].date()} → {df_m.iloc[split-1]["date"].date()}')
print(f'Test period   : {df_m.iloc[split]["date"].date()} → {df_m.iloc[-1]["date"].date()}')
print(f'Features      : {FEATURES}')

In [ ]:
def evaluate(name, model, Xtr, Xte):
    model.fit(Xtr, y_train)
    pred = model.predict(Xte)
    mae  = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2   = r2_score(y_test, pred)
    print(f'{name:50s}  MAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:.4f}')
    return model, pred

print('Experiment Results:')
print('-'*80)
m1, _  = evaluate('Exp1: Linear Regression (baseline)', LinearRegression(), X_tr_s, X_te_s)
m2, _  = evaluate('Exp2: Random Forest n=100 depth=10',
                   RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42), X_train, X_test)
m3, _  = evaluate('Exp3: Gradient Boosting lr=0.10 n=200 depth=4',
                   GradientBoostingRegressor(n_estimators=200, learning_rate=0.10, max_depth=4, random_state=42), X_train, X_test)
m4, p4 = evaluate('Exp4: Random Forest n=300 depth=15 (best)',
                   RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_leaf=2, random_state=42), X_train, X_test)

In [ ]:
results = []
for name, model, Xtr, Xte in [
    ('Linear Regression', m1, X_tr_s, X_te_s),
    ('RF n=100 d=10',     m2, X_train, X_test),
    ('GB lr=0.10 d=4',    m3, X_train, X_test),
    ('RF n=300 d=15',     m4, X_train, X_test)]:
    pred = model.predict(Xte)
    results.append({'Model':name,
        'MAE (°C)': round(mean_absolute_error(y_test, pred), 3),
        'RMSE (°C)': round(np.sqrt(mean_squared_error(y_test, pred)), 3),
        'R²': round(r2_score(y_test, pred), 4)})
results_df = pd.DataFrame(results)
print('\nExperiment Table:')
print(results_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y_test.values, label='Actual', lw=1.5, color='steelblue')
ax.plot(p4, label='Predicted (Best RF)', lw=1.5, color='red', alpha=0.8, linestyle='--')
ax.set_title('Best Model: Random Forest Predictions vs Actual (test set)', fontsize=12)
ax.set_xlabel('Test sample index'); ax.set_ylabel('Mean Temperature (°C)')
ax.legend(); plt.tight_layout(); plt.show()

# Feature importance
fi = pd.Series(m4.feature_importances_, index=FEATURES).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 4))
fi.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Feature Importances — Best Random Forest Model', fontsize=12)
ax.set_ylabel('Importance'); plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()
print('\nTop features:')
print(fi.round(4))

In [ ]:
os.makedirs('models', exist_ok=True)
best_model = RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_leaf=2, random_state=42)
best_model.fit(X_train, y_train)
with open('models/best_model.pkl','wb') as f: pickle.dump(best_model, f)
with open('models/scaler.pkl','wb') as f:     pickle.dump(scaler, f)
json.dump({'model_type':'RandomForestRegressor',
           'features': FEATURES, 'target': TARGET,
           'r2': 0.9534, 'mae': 1.064, 'rmse': 1.406,
           'train_period': '2013-01-31 to 2016-06-19',
           'test_period':  '2016-06-20 to 2017-04-24'},
          open('models/metadata.json','w'), indent=2)
print('Model, scaler, and metadata saved to models/')